In [ ]:
import pandas as pd
df_transactions = pd.read_parquet('ledger_transactions.parquet')


mean_amount = df_transactions.groupby("category")["amount"].transform("mean")
std_amount = df_transactions.groupby("category")["amount"].transform("std")

upper_limit = mean_amount + (3 * std_amount)
is_outlier = df_transactions["amount"] > upper_limit
df_transactions = df_transactions[~is_outlier]

df_transactions['timestamp'] = pd.to_datetime(df_transactions['timestamp'],format='mixed',dayfirst=True, errors='coerce')
df_transactions = df_transactions.sort_values(by=['applicant_id', 'timestamp'])

df_transactions['rolling_volume'] = (
    df_transactions.groupby("applicant_id")
    .rolling('30D', on="timestamp")["amount"]
    .sum()
    .values
)

df_transactions['rolling_velocity'] = (
    df_transactions.groupby("applicant_id")
    .rolling('30D', on="timestamp")["amount"]
    .count()
    .values
)



,transaction_id,applicant_id,timestamp,amount,terminal_location,category,rolling_volume,rolling_velocity
0,TX07828,APP001,2025-10-11 01:26:31,6.032448,Batumi Mall,Retail,6.032448,1.0
1,TX00111,APP001,2025-10-11 05:05:28,170.850972,Batumi Mall,Utilities,176.883421,2.0
2,TX20083,APP001,2025-10-11 05:51:58,327.238677,Online Gateway,Groceries,504.122098,3.0
3,TX10474,APP001,2025-10-11 07:12:37,75.333494,Online Gateway,Retail,579.455591,4.0
4,TX23883,APP001,2025-10-11 07:41:23,86.848236,Tbilisi Central,Retail,666.303827,5.0
5,TX27102,APP001,2025-10-11 08:38:55,375.821445,Tbilisi Central,Retail,1042.125273,6.0
6,TX10585,APP001,2025-10-11 12:52:23,271.938847,Kutaisi Station,Utilities,1314.064120,7.0
7,TX29137,APP001,2025-10-11 13:19:32,506.256021,Batumi Mall,Travel,1820.320141,8.0
8,TX11134,APP001,2025-10-11 18:13:45,19.420773,Batumi Mall,Groceries,1839.740915,9.0
9,TX21088,APP001,2025-10-11 20:53:49,4.929292,Batumi Mall,Utilities,1844.670206,10.0


In [ ]:
import pandas as pd
import json

# read data
df_applications = pd.read_csv('applications.csv')
with open('bureau_scores.json') as f:
    raw_bureau_scores = json.load(f)

# normalize collumn names and clean dupplicates
df_bureau_scores = pd.json_normalize(raw_bureau_scores)
df_applications.columns = df_applications.columns.str.replace(' ', '')

transformed_income = df_applications.groupby("education_level")["monthly_income"].transform('median')
filled_df_applications = df_applications.fillna({'monthly_income': transformed_income})

filled_df_applications["submission_date"]= pd.to_datetime(filled_df_applications['submission_date'], format='mixed',dayfirst=True, errors='coerce')
sorted_applications = filled_df_applications.sort_values(by='submission_date', ascending=False)
drop_duplicates_applications = sorted_applications.drop_duplicates(subset='applicant_id', keep='first')

# join two tables
clean_applications_with_scores = df_bureau_scores.set_index("applicant_id").join(drop_duplicates_applications.set_index("applicant_id"),how='inner')

clean_applications_with_scores.head(20)



,credit_profile.bureau_score,credit_profile.past_defaults,credit_profile.hard_inquiries,full_name,submission_date,monthly_income,education_level,employment_status
applicant_id,,,,,,,,
APP001,720,0,1,Giorgi Beridze,2025-03-09,2500.0,Bachelor,Employed
APP002,580,2,4,Nino Kalandadze,2025-08-09,1800.0,Master,Employed
APP003,640,1,2,Levan Jishkariani,2025-04-09,1200.0,High School,Unemployed
APP004,790,0,0,Mariam Lomidze,2025-05-09,4200.0,PhD,Employed
APP005,550,3,5,Irakli Mgeladze,2025-06-09,2500.0,Bachelor,Self-Employed
APP006,710,0,1,Salome Kankia,2025-07-09,3100.0,Master,Employed
